In [0]:
import json
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
bronze_df=spark.table("workspace.default.cricket_bronze_current_matches")

raw_json=bronze_df.select("raw_json").collect()[0]["raw_json"]
api_data=json.loads(raw_json) #converts json to python dictionaries


matches=api_data.get("data",[])
print("total matches found:",len(matches))
print(matches[0] if len(matches)>0 else "No matches found")


In [0]:
silver_rows=[]

for match in matches:
    teams=match.get("teams",[])
    score=match.get("score",[])

    team_1=teams[0] if len(teams)>0 else None
    team_2=teams[1] if len(teams)>1 else None

    score_1=None
    score_2=None
    # print(team_1,team_2)

    s1_inning=None
    s2_inning=None
    if len(score)>0:
        s1=score[0]
        score_1=f"{s1.get("r",0)}/{s1.get("w",0)} in {s1.get("o",0)} overs"
        s1_inning=s1.get("inning",0)
    if len(score)>1:
        s2=score[1]
        score_2=f"{s2.get("r",0)}/{s2.get("w",0)} in {s2.get("o",0)} overs"
        s2_inning=s2.get("inning",0)

    silver_rows.append({
    "match_id": match.get("id"),
    "match_name": match.get("name"),
    "status": match.get("status"),
    "venue": match.get("venue"),
    "match_date": match.get("date"),
    "date_time_gmt": match.get("dateTimeGMT"),
    "team_1": team_1,
    "team_2": team_2,
    "score_1": f"{s1.get('r')}/{s1.get('w')}" if s1 else None,
    "score_2": f"{s2.get('r')}/{s2.get('w')}" if s2 else None,
    "match_started": match.get("matchStarted"),    # was match_started
    "match_ended": match.get("matchEnded"),        # was match_ended
    "innings_1":s1_inning,                 # from score[], not top level
    "innings_2":s2_inning,
})
print("silver rows prepared:",len(silver_rows))

In [0]:
silver_schema = StructType([
    StructField("match_id", StringType(), True),
    StructField("match_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("venue", StringType(), True),
    StructField("match_date", StringType(), True),
    StructField("date_time_gmt", StringType(), True),
    StructField("team_1", StringType(), True),
    StructField("team_2", StringType(), True),
    StructField("score_1", StringType(), True),
    StructField("score_2", StringType(), True),
    StructField("match_started", BooleanType(), True),
    StructField("match_ended", BooleanType(), True),
    StructField("innings_1", StringType(), True),
    StructField("innings_2", StringType(), True),
])

silver_df = spark.createDataFrame(silver_rows,silver_schema)\
    .withColumn("match_date",to_date(col("match_date")))\
    .withColumn("loaded_at",current_timestamp())

display(silver_df)

In [0]:
silver_df.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema","true")\
    .saveAsTable("workspace.default.cricket_silver_current_matches")

print("SIlver table created succesfully")

In [0]:
%sql
select * from workspace.default.cricket_silver_current_matches